# Hyperparameter Sweep — local, no AWS cost
Grid search over the params most likely to move the needle, evaluated on val MAE.
Winner gets ported into train_entry.py for the real SageMaker training run.

In [1]:
import itertools
import time
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

DATA_DIR = "../data/processed"

FEATURES = [
    "hour", "dayofweek", "month", "quarter", "is_weekend", "is_holiday",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
    "lag_24h", "lag_168h", "rolling_mean_168h", "rolling_std_168h",
]
TARGET = "PJME_MW"  # matches actual Parquet column name (case-sensitive)

In [2]:
train = pd.read_parquet(f"{DATA_DIR}/train.parquet")
val = pd.read_parquet(f"{DATA_DIR}/val.parquet")

X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = val[FEATURES], val[TARGET]

In [3]:
# ---- Search space ----
# Kept deliberately small (24 combos) — each trial takes a similar amount of time
# to your original run (~1 min), so this sweep should finish in ~20-25 min total.
param_grid = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 5],
    "gamma": [0, 0.1],
}

keys = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f"Running {len(combos)} combinations...")

Running 36 combinations...


In [4]:
results = []

for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    start = time.time()

    model = xgb.XGBRegressor(
        n_estimators=1000,           # generous ceiling — early stopping picks the real number
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        min_child_weight=params["min_child_weight"],
        gamma=params["gamma"],
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        early_stopping_rounds=30,
        eval_metric="mae",
    )

    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    val_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, val_pred)
    mape = mean_absolute_percentage_error(y_val, val_pred) * 100
    elapsed = time.time() - start

    results.append({**params, "best_iteration": model.best_iteration, "mae": mae, "mape": mape, "seconds": elapsed})
    print(f"[{i+1}/{len(combos)}] {params} -> MAE={mae:.1f}, MAPE={mape:.2f}%, "
          f"best_iter={model.best_iteration}, {elapsed:.1f}s")

[1/36] {'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 1, 'gamma': 0} -> MAE=1791.0, MAPE=5.75%, best_iter=591, 4.5s
[2/36] {'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 1, 'gamma': 0.1} -> MAE=1791.0, MAPE=5.75%, best_iter=591, 3.0s
[3/36] {'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0} -> MAE=1785.2, MAPE=5.73%, best_iter=823, 3.7s
[4/36] {'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'gamma': 0.1} -> MAE=1785.2, MAPE=5.73%, best_iter=823, 3.3s
[5/36] {'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 1, 'gamma': 0} -> MAE=1792.2, MAPE=5.77%, best_iter=383, 1.5s
[6/36] {'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 1, 'gamma': 0.1} -> MAE=1792.2, MAPE=5.77%, best_iter=383, 1.7s
[7/36] {'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 5, 'gamma': 0} -> MAE=1792.8, MAPE=5.77%, best_iter=423, 1.8s
[8/36] {'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 5, 'gamma': 0.1} -> MAE=

In [5]:
# ---- Results, sorted by MAE ----
results_df = pd.DataFrame(results).sort_values("mae").reset_index(drop=True)
print("\n=== Top 5 configs ===")
print(results_df.head(5).to_string(index=False))

results_df.to_csv("../docs/hyperparameter_sweep_results.csv", index=False)
print("\nSaved full results to docs/hyperparameter_sweep_results.csv")


=== Top 5 configs ===
 max_depth  learning_rate  min_child_weight  gamma  best_iteration         mae     mape  seconds
         8           0.10                 1    0.1              58 1771.789201 5.684493 0.692731
         8           0.10                 1    0.0              58 1771.789201 5.684493 0.699392
         8           0.05                 5    0.1             135 1774.466128 5.687113 1.191577
         8           0.05                 5    0.0             135 1774.466128 5.687113 1.242450
         8           0.05                 1    0.1             104 1774.798295 5.689091 1.054100

Saved full results to docs/hyperparameter_sweep_results.csv


In [ ]:
best = results_df.iloc[0]
print(f"\nBest config: max_depth={int(best['max_depth'])}, "
      f"learning_rate={best['learning_rate']}, "
      f"min_child_weight={int(best['min_child_weight'])}, "
      f"gamma={best['gamma']}")
print(f"Best val MAE: {best['mae']:.1f} MW (vs. previous 1786.4 MW)")
print(f"Best val MAPE: {best['mape']:.2f}% (vs. previous 5.68%)")
print(f"n_estimators to use: {int(best['best_iteration'])} " photo, aadhar, election card, 
      f"(where early stopping landed — add a small buffer, e.g. +20, for the real run)")


Best config: max_depth=8, learning_rate=0.1, min_child_weight=1, gamma=0.1
Best val MAE: 1771.8 MW (vs. previous 1786.4 MW)
Best val MAPE: 5.68% (vs. previous 5.68%)
n_estimators to use: 58 (where early stopping landed — add a small buffer, e.g. +20, for the real run)
